In [ ]:
import os
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel, BertForTokenClassification, Trainer, TrainingArguments
from transformers import DataCollatorForTokenClassification,set_seed
from datasets import Dataset
import torch
from evaluate import load
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score,roc_curve,precision_recall_curve, auc
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score, matthews_corrcoef
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import optuna
import ast

In [ ]:
# Set random seeds for reproducibility of your trainings run
def set_seeds(s):
    torch.manual_seed(s)
    np.random.seed(s)
    random.seed(s)
    set_seed(s)
    
set_seeds(42) 
tokenizer = BertTokenizer.from_pretrained('Rostlab/prot_bert',do_lower_case=False) # vocab.txt
model = BertForTokenClassification.from_pretrained('Rostlab/prot_bert',num_labels=2) #output_attentions=True

In [ ]:
def create_dataset(tokenizer,df,labels):
    seqs_df = df.copy()
    seqs_df['complex_seq']= seqs_df["antibody_seq"]+seqs_df["antigen_seq"]
    seqs_df["complex_seq"] = seqs_df["complex_seq"].str.replace('|'.join(["O", "B", "U", "Z"]), "X", regex=True)
    seqs_df['complex_seq'] = seqs_df.apply(lambda row: " ".join(row["complex_seq"]), axis=1)
    
    tokenized = tokenizer(list(seqs_df["complex_seq"]), max_length=3000, padding=False, truncation=True)
    dataset = Dataset.from_dict(tokenized)
    labels = [[-100]+l[:3000]+[-100] for l in labels] 
    dataset = dataset.add_column("labels", labels)
    dataset = dataset.with_format("torch")
     
    return dataset

In [ ]:
data = pd.read_excel('data/clean_data_labeled.xlsx')
# 填充空值
data['antibody_seq1'] = data['antibody_seq1'].fillna('')
data['antibody_seq2'] = data['antibody_seq2'].fillna('')

# 计算长度
data['antibody_seq1_len'] = data['antibody_seq1'].str.len()
data['antibody_seq2_len'] = data['antibody_seq2'].str.len()

data['antibody_seq'] = data['antibody_seq1']+data['antibody_seq2']

def safe_eval(literal):
    return ast.literal_eval(literal)
data['labels_all'] = data.apply(lambda row: safe_eval(row['antibody_seq1_label']) + safe_eval(row['antibody_seq2_label']) + safe_eval(row['ag_label']), axis=1)

In [ ]:
train = data[data.dataset=='train']
valid = data[data.dataset=='valid']
train_set = create_dataset(tokenizer,train,train['labels_all'])
valid_set = create_dataset(tokenizer,valid,valid['labels_all'])

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  
# from transformers import logging
# logging.set_verbosity(logging.ERROR)

# Step 1.
device = "cuda" if torch.cuda.is_available() else "cpu"
def model_init():
    set_seeds(42)
    model = BertForTokenClassification.from_pretrained('../PLMs/prot_bert',num_labels=2)
#     model.gradient_checkpointing_enable()
    return model.to(device)

# Step 2. 
data_collator = DataCollatorForTokenClassification(tokenizer,padding="longest",label_pad_token_id=-100)

# Step 3. 
def compute_metrics(eval_pred):

    metric = load("../polyreactivity/evaluate/metrics/accuracy")
    predictions, labels = eval_pred

    labels = labels.reshape((-1,))

    predictions = np.argmax(predictions, axis=2)
    predictions = predictions.reshape((-1,))

    predictions = predictions[labels!=-100]
    labels = labels[labels!=-100]

    return metric.compute(predictions=predictions, references=labels)

# Step 4. 
def hp_space_optuna(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 5e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.05, 0.2),
        "lr_scheduler_type": trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"]),
    }

# Step 5. 
def objective(trial):
    params = hp_space_optuna(trial)

    training_args = TrainingArguments(
        output_dir=f"./para_search_tmp",
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        per_device_train_batch_size=1,
        num_train_epochs=3,
        learning_rate=params["learning_rate"],
        weight_decay=params["weight_decay"],
        warmup_ratio=params["warmup_ratio"],
        lr_scheduler_type=params["lr_scheduler_type"],
        load_best_model_at_end=False,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        seed=42,
        fp16=True
    )

    trainer = Trainer(
        model_init=model_init, 
        args=training_args,
        train_dataset=train_set,
        eval_dataset=valid_set,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()
    eval_loss = metrics["eval_loss"]
    return eval_loss

sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)

In [ ]:
# 开始搜索
study.optimize(objective, n_trials=5)

In [ ]:
print("\n=====best params=====")
best_trial = study.best_trial
print(f"Best eval_loss: {best_trial.value:.4f}")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer,padding="longest",label_pad_token_id=-100)
best_params = best_trial.params
final_args = TrainingArguments(
    output_dir="./",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=1,
    num_train_epochs=10,
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"],
    warmup_ratio=best_params["warmup_ratio"],
    lr_scheduler_type=best_params["lr_scheduler_type"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42
)
# Trainer
final_trainer = Trainer(
    model=model,
    args=final_args,
    train_dataset=train_set,
    eval_dataset=valid_set,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

final_trainer.train()

In [ ]:
trained_model = final_trainer.model
trained_model.save_pretrained('./prot_bert_finetuning')

In [ ]:
trained_model.to('cuda')
trained_model.eval()

In [ ]:
def metrics_cal_with_threshold(proba_1_flatten,padded_labels_flatten,threshold=0.5):
    # 计算混淆矩阵
    pred_label = (proba_1_flatten >= threshold).astype(int)
    true_label = padded_labels_flatten
    proba = proba_1_flatten
    tn, fp, fn, tp = confusion_matrix(true_label, pred_label).ravel()
    sensitivity = recall_score(true_label, pred_label)#  敏感性 (Sensitivity, Recall)
    specificity = tn / (tn + fp)                      # 特异性 (Specificity)
    acc = (tn + tp)/(tn + fp + fn + tp)
    f1 = f1_score(true_label, pred_label)             #  F1-score
    mcc = matthews_corrcoef(true_label, pred_label)   # MCC
    
    baseline = np.mean(true_label)
    # AUC类指标必须保证两类都存在
    if len(np.unique(true_label)) < 2:
        AUC = None  # 或者可以设置为0或任意指示值
        pr_auc = None
        fold_improve = None
    else:
        AUC = roc_auc_score(true_label,proba)
        precision, recall, _ = precision_recall_curve(true_label, proba)
        pr_auc = auc(recall, precision)
        fold_improve = pr_auc / baseline if baseline > 0 else None
        
    # 打印结果
    print(f"Sensitivity (SN): {sensitivity}")
    print(f"Specificity (SP): {specificity}")
    print(f"ACC: {acc}")
    print(f"F1-score: {f1}")
    print(f"MCC: {mcc}")
    print(f"ROC_AUC: {AUC}")
    print(f"PR_AUC: {pr_auc}")
    print(f"baseline: {baseline}")
    print(f"PR fold_improve: {fold_improve}")
    return float(sensitivity), float(specificity), float(acc), float(f1), float(mcc), float(AUC),float(pr_auc),float(baseline),float(fold_improve) 

def model_test(dataloader):
#     proba_0=[]
    proba_1=[]
    predictions = []
    padded_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch['input_ids'].to('cuda:0')
            attention_mask = batch['attention_mask'].to('cuda:0')
            padded_labels += batch['labels'].tolist()
#             print(f"Label Length: {batch['labels'][0]}")
            output = trained_model(input_ids, attention_mask=attention_mask)
#             proba_0_batch = output.logits.softmax(dim=-1)[:, :, 0].tolist()
            proba_1_batch = output.logits.softmax(dim=-1)[:, :, 1].tolist()
#             print(f"Proba_1 Length: {len(proba_1_batch[0])}")
            predictions_batch = output.logits.argmax(dim=-1).tolist()
#             print(f"Predictions Length: {len(predictions_batch[0])}")

#             proba_0 += proba_0_batch
            proba_1 += proba_1_batch
            predictions += predictions_batch
    ##将所有预测结果展开，计算总体的性能
    def flatten(l):
        return [item for sublist in l for item in sublist]
    
    # flatten and convert to np array for easy slicing in the next step
    predictions_flatten = np.array(flatten(predictions))
    padded_labels_flatten = np.array(flatten(padded_labels))
    proba_1_flatten = np.array(flatten(proba_1))
    # Filter out all invalid (label = -100) values
    predictions_flatten = predictions_flatten[padded_labels_flatten!=-100]
    proba_1_flatten = proba_1_flatten[padded_labels_flatten!=-100]
    padded_labels_flatten = padded_labels_flatten[padded_labels_flatten!=-100]
    
    return proba_1,predictions,padded_labels,proba_1_flatten,predictions_flatten,padded_labels_flatten

def opimal_threshold(padded_labels,probe_1):
    true_label = padded_labels
    probe = probe_1
    fpr, tpr, thresholds = roc_curve(true_label, probe)
    auc = roc_auc_score(true_label, probe)
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]
    sensitivity = tpr[optimal_idx]
    specificity = 1 - fpr[optimal_idx]
    print("AUC:", auc)
    print("Optimal Threshold:", optimal_threshold)
    print("Sensitivity at Optimal Threshold:", sensitivity)
    print("Specificity at Optimal Threshold:", specificity)
    plt.figure()
    plt.rcParams['pdf.use14corefonts'] = True ##pdf字体在AI中可编辑
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.4f)' % auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic Curve')
    plt.legend(loc="lower right")
    plt.savefig('protbert_optimal_threshold_auc.pdf',format='pdf')
    plt.show()
    return optimal_threshold

In [ ]:
train_dataloader = DataLoader(train_set, batch_size=4, shuffle = False, collate_fn = data_collator)
proba_1,predictions,padded_labels,proba_1_flatten,predictions_flatten,padded_labels_flatten = model_test(train_dataloader)

In [ ]:
threshold = opimal_threshold(padded_labels_flatten,proba_1_flatten)
metrics_cal_with_threshold(proba_1_flatten,padded_labels_flatten,threshold=threshold)

In [ ]:
test = data[data.dataset=='test']
test_set = create_dataset(tokenizer,test,test['labels_all'])
test_dataloader = DataLoader(test_set, batch_size=4, shuffle = False, collate_fn = data_collator)

In [ ]:
_,_,_,proba_1_flatten,_,padded_labels_flatten = model_test(test_dataloader)
metrics_cal_with_threshold(proba_1_flatten,padded_labels_flatten,threshold=threshold)